# LoRA vs Base Model Accuracy Eval — GSM8K (200 problems)

This notebook measures whether the fixed LoRA adapter actually improves math accuracy.

**What it does:**
1. Loads 200 problems from GSM8K (test split)
2. Runs the **base model only** (no LoRA) on all 200, grades answers, saves results
3. Frees GPU memory
4. Runs the **base model + LoRA adapter** on the same 200 problems, grades answers, saves results
5. Compares the two accuracy scores side by side

Run cells top to bottom. Each model-eval cell will print a time estimate for the full run after the first few problems — check it before walking away.


## 1. Setup

In [ ]:
# Point Python at your repo's backend/ folder (the src/ package root)
import sys, os

# EDIT THIS if your repo lives somewhere else
REPO_BACKEND = r"C:\Users\Lenovo\SLM\backend"

sys.path.insert(0, REPO_BACKEND)
os.chdir(REPO_BACKEND)  # some relative paths in the codebase assume this

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM total: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))


In [ ]:
# Install the dataset library if you don't have it yet (safe, pure-python, tiny)
# Uncomment the line below and run once:
# %pip install datasets

from datasets import load_dataset
import re, json, time, gc
from tqdm.auto import tqdm

from src.generation.inference import MathSolverInference
from src.output.formatter import OutputFormatter
from src.input_processing.latex_parser import LaTeXParser
import sympy


## 2. Load 200 GSM8K problems (fixed seed, reproducible)

In [ ]:
gsm8k = load_dataset("gsm8k", "main", split="test")
gsm8k_shuffled = gsm8k.shuffle(seed=42)
eval_set = gsm8k_shuffled.select(range(200))

def extract_gsm8k_answer(answer_field: str) -> float:
    """GSM8K ground-truth answers end with '#### <number>'."""
    match = re.search(r'####\s*([\-\d,\.]+)', answer_field)
    if not match:
        raise ValueError(f"Could not parse ground truth from: {answer_field!r}")
    return float(match.group(1).replace(',', ''))

problems = [
    {"question": row["question"], "ground_truth": extract_gsm8k_answer(row["answer"])}
    for row in eval_set
]

print(f"Loaded {len(problems)} problems")
print("Example:", problems[0])


## 3. Grading logic (reuses your fixed \\boxed{} extraction + LaTeX parser)

In [ ]:
latex_parser = LaTeXParser()

def extract_number(s):
    """Best-effort extraction of a numeric value from a model's final_answer string."""
    if s is None:
        return None
    s = s.strip()
    s = s.replace('$', '').replace(',', '').replace('\\!', '').replace('\\,', '').strip()
    s = s.rstrip('.').strip()

    # 1. Direct float
    try:
        return float(s)
    except ValueError:
        pass

    # 2. LaTeX (e.g. \\frac{1}{2}) -> sympy -> float, using the repo's own
    #    (now-fixed) brace-balanced LaTeX converter
    try:
        converted = latex_parser.parse_latex(s)
        val = sympy.sympify(converted)
        return float(val)
    except Exception:
        pass

    # 3. Fallback: last number-looking substring in the text
    matches = re.findall(r'-?\d+\.?\d*', s)
    if matches:
        try:
            return float(matches[-1])
        except ValueError:
            return None
    return None


def extract_number_robust(predicted_str, fallback_full_text=None):
    """extract_final_answer()'s 'last sentence' fallback can return an
    empty string when the model's text ends in a period and has neither a
    \\boxed{} nor a 'Final Answer:' tag -- found this while testing this
    notebook. If the primary extraction comes up empty, fall back to
    scanning the model's full raw solution text for a trailing number
    instead of grading a possibly-correct answer as wrong."""
    num = extract_number(predicted_str)
    if num is not None:
        return num
    if fallback_full_text:
        return extract_number(fallback_full_text)
    return None


def is_correct(predicted_str, ground_truth_float, fallback_full_text=None, tol=1e-4):
    pred_num = extract_number_robust(predicted_str, fallback_full_text=fallback_full_text)
    if pred_num is None:
        return False
    return abs(pred_num - ground_truth_float) < tol

# Instruct the model explicitly to box its final answer, so grading is reliable
# regardless of which system_prompt style the base checkpoint prefers.
EVAL_SYSTEM_PROMPT = (
    "Solve the math problem below step by step. "
    "Show your reasoning, then give the final numeric answer wrapped in \\boxed{}."
)


## 4. Eval loop (shared by both runs)

In [ ]:
def run_eval(solver, problems, label, out_path, time_check_at=3):
    results = []
    correct = 0
    times = []
    pbar = tqdm(problems, desc=label)

    for i, item in enumerate(pbar):
        t0 = time.time()
        try:
            res = solver.solve(
                item["question"],
                system_prompt=EVAL_SYSTEM_PROMPT,
                max_tokens=512,
                temperature=0.1,   # near-deterministic; HF requires temperature > 0 with do_sample=True
                use_tools=False,   # pure model reasoning -- isolates the LoRA weight effect
            )
            predicted = res["final_answer"]
            solution_text = res["solution"]
            error = None
        except Exception as e:
            predicted = None
            solution_text = None
            error = str(e)

        dt = time.time() - t0
        times.append(dt)
        ok = is_correct(predicted, item["ground_truth"], fallback_full_text=solution_text)
        correct += int(ok)

        results.append({
            "question": item["question"],
            "ground_truth": item["ground_truth"],
            "predicted_raw": predicted,
            "correct": ok,
            "time_sec": round(dt, 2),
            "error": error,
            "raw_solution": solution_text,
        })

        pbar.set_postfix(acc=f"{correct/(i+1):.1%}", avg_s=f"{sum(times)/len(times):.1f}")

        # After a handful of problems, print a time estimate for the full run
        if i + 1 == time_check_at:
            avg = sum(times) / len(times)
            projected_total_min = avg * len(problems) / 60
            print(f"\n[{label}] Avg {avg:.1f}s/problem after {time_check_at} problems. "
                  f"Projected total for {len(problems)} problems: ~{projected_total_min:.1f} minutes.\n")

    accuracy = correct / len(problems)
    with open(out_path, "w") as f:
        json.dump({
            "label": label,
            "accuracy": accuracy,
            "n": len(problems),
            "avg_time_sec": sum(times) / len(times),
            "results": results,
        }, f, indent=2)

    print(f"\n=== {label}: {correct}/{len(problems)} correct = {accuracy:.1%} ===")
    print(f"Saved detailed results to {out_path}")
    return accuracy, results


## 5. Run 1 — Base model only (no LoRA)

`lora_adapter_path=None` skips loading the adapter entirely, so this measures the raw base checkpoint.

In [ ]:
base_solver = MathSolverInference(
    lora_adapter_path=None,
    enable_tools=False,
)

base_accuracy, base_results = run_eval(
    base_solver,
    problems,
    label="base_model",
    out_path="eval_base_results.json",
)


In [ ]:
# Free GPU memory before loading the second model -- important on a 6GB card
del base_solver
gc.collect()
torch.cuda.empty_cache()
print("Freed GPU memory. Current allocated:", torch.cuda.memory_allocated() / 1e9, "GB")


## 6. Run 2 — Base model + LoRA adapter

Uses the default path, which now correctly resolves to `backend/models/lora_adapter/` on any OS.

In [ ]:
lora_solver = MathSolverInference(
    enable_tools=False,
    # lora_adapter_path not passed -> uses the fixed DEFAULT_LORA_ADAPTER_PATH
)

lora_accuracy, lora_results = run_eval(
    lora_solver,
    problems,
    label="lora_model",
    out_path="eval_lora_results.json",
)


In [ ]:
del lora_solver
gc.collect()
torch.cuda.empty_cache()


## 7. Compare results

In [ ]:
print(f"Base model accuracy:       {base_accuracy:.1%}  ({sum(r['correct'] for r in base_results)}/{len(problems)})")
print(f"Base + LoRA accuracy:      {lora_accuracy:.1%}  ({sum(r['correct'] for r in lora_results)}/{len(problems)})")
print(f"Difference:                {(lora_accuracy - base_accuracy)*100:+.1f} percentage points")

# Problems where LoRA fixed a base-model mistake, and vice versa
lora_fixed = [
    (b, l) for b, l in zip(base_results, lora_results)
    if not b["correct"] and l["correct"]
]
lora_broke = [
    (b, l) for b, l in zip(base_results, lora_results)
    if b["correct"] and not l["correct"]
]

print(f"\nLoRA fixed {len(lora_fixed)} problems the base model got wrong")
print(f"LoRA broke {len(lora_broke)} problems the base model got right")

if lora_fixed:
    print("\nExample LoRA fixed:")
    b, l = lora_fixed[0]
    print("Q:", b["question"][:200])
    print("Ground truth:", b["ground_truth"])
    print("Base predicted:", b["predicted_raw"])
    print("LoRA predicted:", l["predicted_raw"])
